<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-05-bigquery-ml/lesson-5.3-llm-in-sql/notebooks/GCP_Capstone_5.3_LLM_SQL.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.3 LLM in SQL — AI.GENERATE, VECTOR_SEARCH & Embeddings
**Netsetos GenAI Engineering — GCP Capstone**

Call Gemini from SQL. Generate embeddings. Run vector search. Build RAG — all without Python.


## Setup


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Ensure this module's datasets exist (idempotent -- BigQuery never auto-creates them).
# NOTE: cells that read rag_data.document_features need Lesson 5.1 run first.
for _ds in ('rag_data', 'ml_models'):
    _d = bigquery.Dataset(f'{PROJECT_ID}.{_ds}'); _d.location = 'US'
    client.create_dataset(_d, exists_ok=True)

def run_query(sql):
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "OK"}')

print(f'Connected to {PROJECT_ID}')


## Cell 1: Create Remote Models


In [ ]:
# Create connection + remote models
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.gemini_flash`
  REMOTE WITH CONNECTION DEFAULT
  OPTIONS (ENDPOINT = 'gemini-3.6-flash')
''')
print('Gemini model created')

run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.embed_005`
  REMOTE WITH CONNECTION DEFAULT
  OPTIONS (ENDPOINT = 'text-embedding-005')
''')
print('Embedding model created')


## Fixture: doc_chunks — the table the lesson queries

In [ ]:
# Fixture: build rag_data.doc_chunks (the table this lesson teaches) from
# Lesson 5.1's rag_data.document_features, so the notebook uses the same
# chunk_id / chunk_text / doc_type column names as the main lesson HTML.
# Prerequisite: run Lesson 5.1 first (BigQuery tables persist per project).
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.doc_chunks` AS
SELECT
  ROW_NUMBER() OVER () AS chunk_id,
  doc_id,
  title         AS chunk_text,
  document_type AS doc_type
FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('doc_chunks fixture ready')

## Cell 2: AI.GENERATE — Summarize Documents


In [ ]:
# Summarize document chunks with AI.GENERATE (Gemini runs per row, in SQL).
# endpoint => 'gemini-3.6-flash' pins the model. Without it, the scalar
# AI.GENERATE falls back to BigQuery's default endpoint instead of the
# gemini_flash remote model registered above.
results = run_query(rf'''
SELECT
  chunk_id, chunk_text,
  AI.GENERATE(
    CONCAT('Summarize in one sentence:\n', chunk_text),
    endpoint => 'gemini-3.6-flash'
  ).result AS summary
FROM `{PROJECT_ID}.rag_data.doc_chunks`
LIMIT 10
''')
print(results)

## Cell 3: ML.GENERATE_EMBEDDING — Text to Vectors


In [ ]:
# Generate embeddings for the chunks (RETRIEVAL_DOCUMENT task type).
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.chunk_embeddings` AS
SELECT
  chunk_id, doc_id, chunk_text,
  ml_generate_embedding_result AS embedding,
  ml_generate_embedding_status AS status
FROM ML.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.ml_models.embed_005`,
  (SELECT chunk_id, doc_id, chunk_text, chunk_text AS content
   FROM `{PROJECT_ID}.rag_data.doc_chunks`),
  STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type)
)
WHERE ml_generate_embedding_status = ''
''')
print('Embeddings generated')

# Verify dimensions
result = run_query(f'''
SELECT chunk_id, ARRAY_LENGTH(embedding) AS dims
FROM `{PROJECT_ID}.rag_data.chunk_embeddings`
LIMIT 5
''')
print(result)

## Cell 4: VECTOR_SEARCH — Find Similar Documents


In [ ]:
# Search for chunks similar to a query
results = run_query(f'''
SELECT
  base.chunk_id, base.chunk_text, distance
FROM VECTOR_SEARCH(
  TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`,
  'embedding',
  (SELECT ml_generate_embedding_result AS embedding
   FROM ML.GENERATE_EMBEDDING(
     MODEL `{PROJECT_ID}.ml_models.embed_005`,
     (SELECT 'machine learning model training' AS content),
     STRUCT('RETRIEVAL_QUERY' AS task_type))),
  top_k => 5,
  distance_type => 'COSINE'
)
ORDER BY distance
''')
print('=== Vector Search Results ===')
print(results)

## Cell 5: RAG-in-SQL — Complete Pipeline


In [ ]:
# Complete RAG: embed query -> vector search -> generate answer
result = run_query(rf'''
SELECT ml_generate_text_llm_result AS answer
FROM ML.GENERATE_TEXT(
  MODEL `{PROJECT_ID}.ml_models.gemini_flash`,
  (SELECT CONCAT(
     'Answer using ONLY this context:\n',
     STRING_AGG(
       FORMAT('[Source %d] %s', base.chunk_id, base.chunk_text),
       '\n'),
     '\n\nQuestion: ', MAX(query.q)
   ) AS prompt
   FROM VECTOR_SEARCH(
     TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`,
     'embedding',
     (SELECT ml_generate_embedding_result AS embedding, content AS q
      FROM ML.GENERATE_EMBEDDING(
        MODEL `{PROJECT_ID}.ml_models.embed_005`,
        (SELECT 'What documents are about machine learning?' AS content),
        STRUCT('RETRIEVAL_QUERY' AS task_type))),
     top_k => 3,
     distance_type => 'COSINE')),
  STRUCT(1024 AS max_output_tokens, 0.2 AS temperature,
        TRUE AS flatten_json_output))
''')
print('=== RAG Answer ===')
print(result.iloc[0]['answer'] if len(result) > 0 else 'No result')

## Cell 6: Stored Procedure — ask_documind()


In [ ]:
# Create the ask_documind stored procedure
run_ddl(rf'''
CREATE OR REPLACE PROCEDURE `{PROJECT_ID}.ml_models.ask_documind`(
  user_question STRING)
BEGIN
  SELECT ml_generate_text_llm_result AS answer
  FROM ML.GENERATE_TEXT(
    MODEL `{PROJECT_ID}.ml_models.gemini_flash`,
    (SELECT CONCAT(
       'Answer from context only:\n',
       STRING_AGG(FORMAT('[%d] %s', base.chunk_id, base.chunk_text), '\n'),
       '\n\nQ: ', MAX(query.q)
     ) AS prompt
     FROM VECTOR_SEARCH(
       TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`, 'embedding',
       (SELECT ml_generate_embedding_result AS embedding, content AS q
        FROM ML.GENERATE_EMBEDDING(
          MODEL `{PROJECT_ID}.ml_models.embed_005`,
          (SELECT user_question AS content),
          STRUCT('RETRIEVAL_QUERY' AS task_type))),
       top_k => 3, distance_type => 'COSINE')),
    STRUCT(1024 AS max_output_tokens, TRUE AS flatten_json_output));
END
''')
print('ask_documind() procedure created')

# Test it
try:
    result = run_query(f"CALL `{PROJECT_ID}.ml_models.ask_documind`('What are the ML topics?')")
    print(result)
except Exception as e:
    print(f'Note: {e}')

## Cell 7: Bulk Operations — Classify + Extract


In [ ]:
# Zero-shot classification with AI.GENERATE (endpoint pins gemini-3.6-flash)
results = run_query(rf'''
SELECT
  chunk_id, chunk_text,
  AI.GENERATE(
    CONCAT('Classify as: research_paper, invoice, legal, or form.\n\nText: ', chunk_text),
    endpoint => 'gemini-3.6-flash'
  ).result AS predicted_type,
  doc_type AS actual_type
FROM `{PROJECT_ID}.rag_data.doc_chunks`
''')
print('=== LLM Classification vs Actual ===')
print(results[['chunk_id','chunk_text','predicted_type','actual_type']])

## ✅ Lesson 5.3 Complete! Module 5 Complete!

**Three AI functions mastered:**
- ✅ AI.GENERATE — Gemini in every row (summarize, classify, extract, translate)
- ✅ ML.GENERATE_EMBEDDING — text to 768-dim vectors in SQL
- ✅ VECTOR_SEARCH — nearest neighbors with COSINE distance + IVF index
- ✅ RAG-in-SQL — complete pipeline in one statement
- ✅ ask_documind() stored procedure

**Module 5 Complete — 3 Lessons:**
- 5.1: CREATE MODEL (LINEAR_REG, LOGISTIC_REG, KMEANS)
- 5.2: ARIMA_PLUS (forecast, anomaly, decompose)
- 5.3: AI functions (AI.GENERATE, VECTOR_SEARCH, embeddings)

**Next: Module 6 — Function Calling & Tool Use**
